## INIT


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace2.silver")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, upper, abs

In [0]:
## Create tables dataframe  

customer_df = spark.table("workspace2.bronze.customers")

products_df = spark.table("workspace2.bronze.products")

sales_df = spark.table("workspace2.bronze.sales")

cust_az_df = spark.table("workspace2.bronze.cust_az")

loc_a1_df = spark.table("workspace2.bronze.loc_a1")

px_cat_df = spark.table("workspace2.bronze.px_cat")

# COSTUMERS

## Dedaplicated

In [0]:
## Remove the duplicated rows from the customer table

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("cst_id") \
    .orderBy(col("cst_create_date").desc())

customer_silver = customer_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

customer_silver = customer_silver.filter(
    (col("cst_id").isNotNull()) &
    (col("row_num") == 1)
)

customer_silver = customer_silver.drop("row_num")

In [0]:
## verify the duplicates and non nulls for the customer id in silver modification also

customer_silver.groupBy("cst_key").count().filter((col("count") > 1) | (col("cst_key").isNull())).display()

## Trimming 


In [0]:
## Trimming the string columns in customer_silver

import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, upper, initcap
from pyspark.sql.types import StringType

for field in customer_silver.schema.fields:
    if isinstance(field.dataType, StringType):
        customer_silver = customer_silver.withColumn(
            field.name, trim(col(field.name))
            )

In [0]:
customer_silver.display()

## Normalazing

In [0]:
customer_silver = (customer_silver
.withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
    .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
    .otherwise("n/a")
)
.withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
    .when(F.upper(F.col("cst_gndr")) == "M", "Male")
    .otherwise("n/a")
)
)

customer_silver.display()



In [0]:
from pyspark.sql.functions import col

customer_silver.select("cst_gndr").distinct().display()
customer_silver.select("cst_marital_status").distinct().display()

## Renaming the columns names

In [0]:
## mapping a dictionnary for the new column names

map_name = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "creation_date"
}


In [0]:
## renaming the columns name

for old_name, new_name in map_name.items():
    customer_silver = customer_silver.withColumnRenamed(old_name, new_name)



In [0]:
customer_silver.display()

## Ingestion to silver layer

In [0]:
## Create silver schema

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace2.silver")

In [0]:
customer_silver.write.mode("overwrite").saveAsTable("workspace2.silver.customer")
customer_silver = spark.table("workspace2.silver.customer")
customer_silver.display()

# PRODUCTS

## Split the prd_key column


In [0]:
from pyspark.sql.functions import col, substring, regexp_replace, when, trim , upper


In [0]:
## Create a new column cat_id from the prd_key column

products_silver = products_df.withColumn(
    "cat_id",
    regexp_replace(substring(col("prd_key"), 1, 5), "-", "_")
)

products_silver.display()

In [0]:
## split the prd_key column to fit the sls_prd_key in sales table

products_silver = products_silver.withColumn(
    "prd_key",
    substring(col("prd_key"), 7, 100)
    )

products_silver.display()

## Replace Null values

In [0]:


products_silver =  products_silver.withColumn(
    "prd_cost",
    when(col("prd_cost").isNull(), 0).otherwise(col("prd_cost"))
)

products_silver.display()

## Normalization 

In [0]:
products_silver = products_silver.withColumn(
    "prd_line",
    when(upper(trim(col("prd_line"))) == 'M', 'Mountain')
    .when(upper(trim(col("prd_line"))) == 'R', 'Road')
    .when(upper(trim(col("prd_line"))) == 'S', 'Other Sales')
    .when(upper(trim(col("prd_line"))) == 'T', 'Touring')
    .otherwise('n/a')
)

products_silver.display()   # calling display() as a method, but NOT assigning

## Generate product end date 

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Define the window: partition by product, order by start date
window_spec = Window.partitionBy("prd_key").orderBy("prd_start_dt")

products_silver = products_silver.withColumn(
    "prd_start_dt",
    F.col("prd_start_dt").cast("date")
).withColumn(
    "prd_end_dt",
    F.date_sub(
        F.lead("prd_start_dt").over(window_spec),
        1
    ).cast("date")
)

## Renaming the columns names

In [0]:
## mapping a dictionnary for the new column names

rename_map = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date",
    "cat_id": "category_id"
}

In [0]:
# Replace the columns names

for old_name, new_name in rename_map.items():
    products_silver = products_silver.withColumnRenamed(old_name, new_name)

In [0]:
products_silver.display()

## Reorder columns 

In [0]:
products_silver = products_silver.select(
    "product_id",
    "product_key",
    "category_id",
    "product_name",
    "product_cost",
    "product_line",
    "product_start_date",
    "product_end_date"
)
products_silver.display()

## Ingestion to silver layer

In [0]:
products_silver.write.mode("overwrite").saveAsTable("workspace2.silver.products")


In [0]:
products_silver = spark.table("workspace2.silver.products")
products_silver.display()

# SALES

## Validating Date

In [0]:
from pyspark.sql.functions import col, when, length
from pyspark.sql import functions as F

# Change the data type of sls_order_dt column

sales_silver = sales_df.withColumn(
    "sls_order_dt",
    F.when(
        (F.col("sls_order_dt") == "0") | (F.length(F.col("sls_order_dt")) != 8),
        F.lit(None).cast("date")
    ).otherwise(
        F.to_date(F.col("sls_order_dt"), "yyyyMMdd")
    )
)

# Change the data type of sls_ship_dt column

sales_silver = sales_silver.withColumn(
    "sls_ship_dt",
    F.when(
        (F.col("sls_ship_dt") == "0") | (F.length(F.col("sls_ship_dt")) != 8),
        F.lit(None).cast("date")
    ).otherwise(
        F.to_date(F.col("sls_ship_dt"), "yyyyMMdd")
    )
)

# Change the data type of sls_due_dt column

sales_silver = sales_silver.withColumn(
    "sls_due_dt",
    F.when(
        (F.col("sls_due_dt") == "0") | (F.length(F.col("sls_due_dt")) != 8),
        F.lit(None).cast("date")
    ).otherwise(
        F.to_date(F.col("sls_order_dt"), "yyyyMMdd")
    )
)

sales_silver.display()

## Data Validation & Repair(sales, price)

In [0]:
# Repairing the missing values in the sls_sales column


sales_silver = sales_silver.withColumn(
    "sls_sales",
    F.when(
        F.col("sls_sales").isNull()
        | (F.col("sls_sales") <= 0)
        | (F.col("sls_sales") != F.col("sls_quantity") * F.abs(F.col("sls_price"))),
        F.col("sls_quantity") * F.abs(F.col("sls_price"))
    ).otherwise(
        F.col("sls_sales")
    )
)

sales_silver.filter(
    (col("sls_sales") != (col("sls_quantity") * abs(col("sls_price"))))
).display()

sales_silver.printSchema()

In [0]:
# Repairing the missing values in the sls_price column

sales_silver = sales_silver.withColumn(
    "sls_price",
    F.when(
        F.col("sls_price").isNull()
        | (F.col("sls_price") <= 0),
        F.col("sls_sales") / F.when(F.col("sls_quantity") == 0, F.lit(None)).otherwise(F.col("sls_quantity"))
    ).otherwise(
        F.col("sls_price")
    )
)

sales_silver.display()

## Rename columns 

In [0]:

rename_map = {
    "sls_ord_num": "order_id",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

In [0]:
## Replacing columns names

for old_name, new_name in rename_map.items():
    sales_silver = sales_silver.withColumnRenamed(old_name, new_name)

In [0]:
sales_silver.display()

## Ingestion to silver layer

In [0]:
sales_silver.write.mode("overwrite").saveAsTable("workspace2.silver.sales")
sales_silver = spark.table("workspace2.silver.sales")
sales_silver.display()

# CUST_AZ

## Transformation CID column

In [0]:
from pyspark.sql.functions import col, substring, regexp_replace, regexp_like, when, trim , upper, length
from pyspark.sql import functions as F

# Transform the CID column to fit the customer key format

cust_az_silver = cust_az_df.withColumn(
    "cid",
    F.when(
        F.col("cid").startswith("NAS"),
        F.substring(F.col("cid"), 4, 1000)
    ).otherwise(F.col("cid"))
)

cust_az_silver.display()

## Validation of BDATE column


In [0]:
# Delete the unvalid birthdate

cust_az_silver = cust_az_silver.withColumn(
    "BDATE",
    F.when(
        F.col("BDATE") >= F.now(), F.lit(None)
    ).otherwise(F.col("BDATE")
)
)

cust_az_silver.display()

In [0]:
cust_az_silver.filter(col("BDATE") >= F.now()).display()

## Normalization

In [0]:
# Standarize the gender

cust_az_silver = cust_az_silver.withColumn(
    "GEN",
    F.when(
        (F.upper(F.col("GEN")) == "M") | 
        (F.upper(F.col("GEN")) == "MALE"), "Male"
    )
    .when(
        (F.upper(F.col("GEN")) == "F") | 
        (F.upper(F.col("GEN")) == "FEMALE"), "Female"
    ).otherwise("n/a")
)

In [0]:
cust_az_silver.select("GEN").distinct().display()

## Rename columns 

In [0]:
# Create the mapping dictionary

rename_map = {
    "CID": "customer_key",
    "BDATE": "birth_date",
    "GEN": "gender"
}

# Rename the columns

for old_name, new_name in rename_map.items():
    cust_az_silver = cust_az_silver.withColumnRenamed(old_name, new_name)


In [0]:
cust_az_silver.display()

## Ingestion to silver layer

In [0]:
cust_az_silver.write.mode("overwrite").saveAsTable("workspace2.silver.cust_az")
cust_az_silver = spark.table("workspace2.silver.cust_az")
cust_az_silver.display()

# LOC_A1

## Transformation CID column

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, substring, regexp_replace

# Replace the character to fit the customer key

loc_a1_silver = loc_a1_df.withColumn(
    "CID",
    F.regexp_replace(col("CID"), '-', '')

)


In [0]:
loc_a1_silver.display()

## Normalization

In [0]:

from pyspark.sql.functions import col, when, trim

loc_a1_silver = loc_a1_silver.withColumn(
    "CNTRY",
    when(trim(col("CNTRY")).isin("USA", "US"), "United States")
    .when(trim(col("CNTRY")) == "DE", "Germany")
    .when(
        (trim(col("CNTRY")) == "") |
        (col("CNTRY").isNull()),
        "n/a"
    )
    .otherwise(trim(col("CNTRY")))
)

loc_a1_silver.display()


In [0]:
loc_a1_silver.select("CNTRY").distinct().display()

## Renaming the columns


In [0]:
loc_a1_silver = loc_a1_silver.withColumnRenamed("CID", "customer_key").withColumnRenamed("CNTRY", "country")
loc_a1_silver.display()


## Ingestion to silver layer

In [0]:
loc_a1_silver.write.mode("overwrite").saveAsTable("workspace2.silver.loc_a1")
loc_a1_silver = spark.table("workspace2.silver.loc_a1")
loc_a1_silver.display()

# PX_CAT

In [0]:
px_cat_silver = px_cat_df

In [0]:
px_cat_silver.display()

## Ingestion to silver layer

In [0]:
px_cat_silver.write.mode("overwrite").saveAsTable("workspace2.silver.px_cat")
px_cat_silver = spark.table("workspace2.silver.px_cat")
px_cat_silver.display()